In [5]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [6]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
import json


def generate_dataset():
    prompt = """
프롬프트 성능 평가를 위한 테스트용 데이터셋을 생성해줘. 
이 데이터셋은 AWS 관련 작업(Python 코드, JSON 설정, 정규표현식 생성)을 수행하는 프롬프트들의 능력을 측정하는 데 사용될 거야.
각 객체가 Python, JSON 또는 정규표현식(Regex)이 필요한 개별 작업을 나타내는 JSON 객체 배열을 만들어줘.

출력 형식 예시:
```json
[
    {
        "task": "작업에 대한 설명",
    },
    ...추가 항목
]
```

* 지침:
1. 단일 Python 함수, 단일 JSON 객체 또는 하나의 정규 표현식으로 해결 가능한 작업에 집중할 것.
2. 코드를 너무 많이 작성하지 않아도 되는 비교적 단순한 작업 위주로 구성할 것.
3. AWS 서비스(S3, EC2, Lambda 등)와 관련된 실무적인 시나리오를 반영할 것.

총 3개의 객체를 생성해줘.
"""
    messages = []
    add_user_message(messages, prompt)
    text = chat(messages, system = """
    당신은 오직 원시 JSON 데이터만 반환하는 시스템입니다. 
    어떠한 인삿말, 설명, 머리글, 바닥글도 작성하지 마십시오.
    특히 마크다운 코드 블록 기호(```json)조차 쓰지 말고, 오직 [ ]로 시작해서 ]로 끝나는 JSON 배열만 출력하십시오.
    """,
                stop_sequences=["```"])
    return json.loads(text)